# Feature Demo: Query Library (API + CLI)

This notebook demonstrates the same query workflows through both interfaces:

- Python API (`battinfo.api`)
- CLI (`battinfo query ...`) via `Typer` test runner

The goal is to make onboarding and debugging interface parity straightforward.


In [ ]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
assert (ROOT / 'src').exists(), f'Repo root not found from {Path.cwd()}'
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from battinfo.api import query_cell_types, query_cell_instances, query_datasets
from battinfo.cli import app
from typer.testing import CliRunner

runner = CliRunner()
print('ROOT:', ROOT)


## 1) Query Cell Types

In [ ]:
# API
api_cell_types = query_cell_types(
    chemistry='LFP',
    format='prismatic',
    nominal_capacity_min=50,
    limit=10,
)
print('API count:', len(api_cell_types))
for r in api_cell_types[:5]:
    print('-', r['id'], '|', r['manufacturer'], '|', r['model_name'])

# CLI
cli_res = runner.invoke(
    app,
    [
        'query',
        'cell-types',
        '--chemistry',
        'LFP',
        '--cell-format',
        'prismatic',
        '--nominal-capacity-min',
        '50',
        '--limit',
        '10',
        '--format',
        'json',
    ],
)
assert cli_res.exit_code == 0, cli_res.stdout
cli_payload = json.loads(cli_res.stdout)
print('CLI count:', cli_payload['count'])

print('parity count match:', len(api_cell_types) == cli_payload['count'])


## 2) Query Cell Instances

In [ ]:
# API
api_instances = query_cell_instances(has_dataset=True, limit=20)
print('API count:', len(api_instances))
for r in api_instances:
    print('-', r['id'], 'dataset=', r.get('dataset_id'))

# CLI
cli_res = runner.invoke(
    app,
    [
        'query',
        'cell-instances',
        '--has-dataset',
        'true',
        '--format',
        'json',
    ],
)
assert cli_res.exit_code == 0, cli_res.stdout
cli_payload = json.loads(cli_res.stdout)
print('CLI count:', cli_payload['count'])
print('parity count match:', len(api_instances) == cli_payload['count'])


## 3) Query Datasets

In [ ]:
cell_id = 'https://w3id.org/battinfo/cell/3m6k-9t2p-7x4h-9nq8'

# API
api_datasets = query_datasets(related_cell_id=cell_id, limit=20)
print('API count:', len(api_datasets))
for r in api_datasets:
    print('-', r['id'], '|', r['title'])

# CLI
cli_res = runner.invoke(
    app,
    [
        'query',
        'datasets',
        '--related-cell-id',
        cell_id,
        '--format',
        'json',
    ],
)
assert cli_res.exit_code == 0, cli_res.stdout
cli_payload = json.loads(cli_res.stdout)
print('CLI count:', cli_payload['count'])
print('parity count match:', len(api_datasets) == cli_payload['count'])


## Debug tips

- If API/CLI counts differ, inspect filters and defaults (`limit`, `offset`, optional flags).
- Use `--format json` in CLI for deterministic comparisons in notebooks/tests.
- Prefer API in production code; use CLI for operations and reproducible manual runs.
